# Update the three SAMap-derived columns in the final fibroblast/decidual objects

SAMap's homology search was rerun on 14 Nov 2026. Here we replace the three columns that depend on it,
compare old with new values, and save new files. Inputs are only read in.

In [5]:
import os
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

In [1]:
human_path = "/n/lab_storage/eddy_lab/Lab/mckinley/data/Cevrim_Human_only_fibdec_cells.h5ad"
human_new_path = "/n/lab_storage/eddy_lab/Lab/mckinley/data/20261114_Human_only_eFib_dec_final_nick.h5ad"
human_out = "/n/lab_storage/eddy_lab/Lab/mckinley/data/Cevrim_Human_only_fibdec_cells_20261114.h5ad"

mouse_path = "/n/lab_storage/eddy_lab/Lab/mckinley/data/Cevrim_XMens_only_fibdec_cells.h5ad"
mouse_new_path = "/n/lab_storage/eddy_lab/Lab/mckinley/data/20261114_mouse_only_eFib_dec_final_nick.h5ad"
mouse_out = "/n/lab_storage/eddy_lab/Lab/mckinley/data/Cevrim_XMens_only_fibdec_cells_20261114.h5ad"

assert not os.path.exists(human_out) and not os.path.exists(mouse_out), "the new files already exist"

In [2]:
human = ad.read_h5ad(human_path)
human_new = ad.read_h5ad(human_new_path, backed="r").obs
assert human.obs_names.equals(human_new.index)

old = human.obs["transferred_mouse_fibdec"]
new = human_new["transferred_mouse_fibdec"]
print("transferred_mouse_fibdec: same label on", f"{(old.astype(str) == new.astype(str)).mean():.2%}", "of nuclei")
human.obs["transferred_mouse_fibdec"] = new

old = human.obs["transferred_mouse_pseudotime"]
new = human_new["transferred_mouse_pseudotime"]
print("transferred_mouse_pseudotime: Pearson r", round(old.corr(new), 4))
human.obs["transferred_mouse_pseudotime"] = new

human.write_h5ad(human_out, compression="gzip", convert_strings_to_categoricals=False)

transferred_mouse_fibdec: same label on 98.75% of nuclei
transferred_mouse_pseudotime: Pearson r 0.9949


In [3]:
mouse = ad.read_h5ad(mouse_path)
mouse_new = ad.read_h5ad(mouse_new_path, backed="r").obs
assert mouse.obs_names.equals(mouse_new.index)

old = mouse.obs["Ulrich2024_sg_Stro7_top50_transferred"]
new = mouse_new["Ulrich2024_sg_Stro7_top50_transferred"]
print("Ulrich2024_sg_Stro7_top50_transferred: Pearson r", round(old.corr(new), 4))
mouse.obs["Ulrich2024_sg_Stro7_top50_transferred"] = new

mouse.write_h5ad(mouse_out, compression="gzip", convert_strings_to_categoricals=False)

Ulrich2024_sg_Stro7_top50_transferred: Pearson r 0.9906


## Read back

The new files hold the new values in the three columnns. all other obs columns are as before.

In [8]:
def same(a, b):
    """True if two stored values are identical: tables, sparse or dense arrays, or dictionaries of them."""
    if isinstance(a, dict):
        return a.keys() == b.keys() and all(same(a[key], b[key]) for key in a)
    if isinstance(a, pd.DataFrame):
        return a.equals(b)
    if sparse.issparse(a):
        return a.shape == b.shape and (a != b).nnz == 0
    a, b = np.asarray(a), np.asarray(b)
    return a.dtype == b.dtype and np.array_equal(a, b, equal_nan=a.dtype.kind == "f")


# chekc the original human object against the updated one in memory
old = ad.read_h5ad(human_path)
others = old.obs.columns.drop(["transferred_mouse_fibdec", "transferred_mouse_pseudotime"])

print("human")
print("nuclei:", old.obs_names.equals(human.obs_names))
print("genes:", old.var_names.equals(human.var_names))
print("X:", same(old.X, human.X))
print("layers:", same(dict(old.layers), dict(human.layers)))
print("obsm:", same(dict(old.obsm), dict(human.obsm)))
print("obsp:", same(dict(old.obsp), dict(human.obsp)))
print("varm:", same(dict(old.varm), dict(human.varm)))
print("varp:", same(dict(old.varp), dict(human.varp)))
print("var:", same(old.var, human.var))
print("uns:", same(dict(old.uns), dict(human.uns)))
print("raw:", old.raw is None and human.raw is None)
print("obs, same column names:", list(old.obs.columns) == list(human.obs.columns))
print("obs, all other columns:", same(old.obs[others], human.obs[others]))

# chekc the original mouse object against the updated one in memory
old = ad.read_h5ad(mouse_path)
others = old.obs.columns.drop(["Ulrich2024_sg_Stro7_top50_transferred"])

print("mouse")
print("nuclei:", old.obs_names.equals(mouse.obs_names))
print("genes:", old.var_names.equals(mouse.var_names))
print("X:", same(old.X, mouse.X))
print("layers:", same(dict(old.layers), dict(mouse.layers)))
print("obsm:", same(dict(old.obsm), dict(mouse.obsm)))
print("obsp:", same(dict(old.obsp), dict(mouse.obsp)))
print("varm:", same(dict(old.varm), dict(mouse.varm)))
print("varp:", same(dict(old.varp), dict(mouse.varp)))
print("var:", same(old.var, mouse.var))
print("uns:", same(dict(old.uns), dict(mouse.uns)))
print("raw:", old.raw is None and mouse.raw is None)
print("obs, same column names:", list(old.obs.columns) == list(mouse.obs.columns))
print("obs, all other columns:", same(old.obs[others], mouse.obs[others]))

human
nuclei: True
genes: True
X: True
layers: True
obsm: True
obsp: True
varm: True
varp: True
var: True
uns: True
raw: True
obs, same column names: True
obs, all other columns: True
mouse
nuclei: True
genes: True
X: True
layers: True
obsm: True
obsp: True
varm: True
varp: True
var: True
uns: True
raw: True
obs, same column names: True
obs, all other columns: True
